## 01: Cohort Assembly

Builds the MDD, BD and non-clinical control cohorts (NCC; `HC` in the code) and their per-task
eligibility.

- **Cognition:** EtM scores and QC flags for every participant who completed any task; only the
  first administration of each task is kept.
- **MDD:** ≥1 MDD record on or before the test date, and no record of bipolar disorder,
  schizophrenia-spectrum disorder, ADHD, autism, drug dependence, dementia or cognitive
  impairment. **BD:** ≥1 BD record on or before the test date and none of the same exclusions
  (MDD records allowed). **HC:** no record of any mental disorder or cognitive impairment.
  Nicotine and tobacco dependence are allowed in all groups.
- **Per task, MDD and BD:** no mood-disorder code within 90 days of the test, in either
  direction; for the wearable analysis, Fitbit recording from at least 90 days before the test
  and no mood-disorder code in the 90 days before the wearable window.
- **Prior episodes** (descriptive only).

**Inputs:** EtM tables and EHR condition records (BigQuery, CDR v9); Fitbit coverage dates
from the wearable cache built by `00_build_cache`.
**Outputs:** `cohort/cohort_master.parquet` (one row per participant) and the attrition log
`cohort/attrition.parquet`.

## Task 0: Setup

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
from functools import reduce
from collections import Counter
from matplotlib import pyplot as plt
from tqdm.auto import tqdm
%matplotlib inline

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.utils import *

# -- Seed incremental attrition log -------------------------------------------
# Rows whose step label starts with one of this notebook's prefixes are dropped and
# regenerated below; rows written by other notebooks are kept, so re-running one
# notebook neither duplicates its own rows nor wipes the rest of the cascade.
NB_ATTRITION_PREFIXES = ('1. ', '3a. ', '3b. ', '3c. ', '4a. ', '4b. ', '4c. ')

attrition_rows.clear()
if gcs_exists('cohort/attrition.parquet'):
    _atr = load_parquet('cohort/attrition.parquet')
    _own = (_atr['step'].astype(str)
                        .map(lambda s: s.startswith(NB_ATTRITION_PREFIXES))
                        .astype(bool))
    attrition_rows.extend(_atr[~_own].to_dict('records'))
    print(f'Attrition log: kept {len(attrition_rows)} step(s) from other notebooks; '
          f'dropped {int(_own.sum())} own row(s) for regeneration.')

# OVERWRITE forces re-acquisition of the expensive cached BigQuery pulls below.
# Derived outputs (cohort_master, attrition) always save — see the save cell.
OVERWRITE = False

# -- Temporal parameters ------------------------------------------------------
N_WASH_DAYS      = 90    # bidirectional washout buffer (days)
N_DAYS_WEARABLE  = 90    # calendar length of wearable feature window
EPISODE_GAP_DAYS = 180   # gap threshold for collapse_episodes() — general episode history
EPISODE_PSYCHOSIS_GAP_DAYS = 60  # gap threshold for collapsing psychosis-specific episodes

# -- OMOP concept IDs ---------------------------------------------------------
MDD_CONCEPTS = (4282096, 4282316)   # MDD single episode, MDD recurrent

DISQUALIFYING_CONCEPTS = (
    436665,   # Bipolar disorder
    443237,   # Manic episode
    435783,   # Schizophrenia
    4286201,  # Schizoaffective disorder
    438409,   # ADHD
    439776,   # Autism spectrum disorder
    440069,   # Drug dependence
    4182210,  # Dementia
    443432,   # Cognitive impairment
)

SUD_EXCEPTIONS           = (4209423, 437264)   # nicotine, tobacco -- NOT disqualifying
MENTAL_DISORDER_ANCESTOR = 432586
HC_EXTRA_EXCLUSIONS      = (440069, 443432)    # not under 432586 in this CDR

# BD qualifying codes; BD_DISQ is DISQUALIFYING minus BD codes themselves
BD_CONCEPTS         = (436665, 443237)
BD_EPISODE_CONCEPTS = (4282096, 4282316, 436665, 443237)  # MDD + BD for washout and episode history
BD_DISQ_CONCEPTS    = tuple(c for c in DISQUALIFYING_CONCEPTS if c not in set(BD_CONCEPTS))

# Psychosis episode concept ancestors (verified against CDR v9 hierarchy)
PSYCHOSIS_DEP_ANCESTOR  = 37111697          # Major depression with psychotic features (unipolar tree)
PSYCHOSIS_BP_ANCESTOR   = 4195158           # Severe bipolar disorder with psychotic features tree
PSYCHOSIS_BP_LEGACY_IDS = (439256, 439246, 35622934)  # legacy BP psychosis codes not under 4195158

# -- EtM schema ---------------------------------------------------------------
# Scores and QC flags live inside the `outcomes` struct; accessed via dot notation.
# emorecog score=None: its outcome (avg_accuracy) is derived after the pull from the
# four per-emotion accuracies.
ETM_TASKS = ['gradcpt', 'delaydiscounting', 'flanker', 'emorecog']

ETM_SCHEMA = {
    'gradcpt': {
        'score':        'dprime',
        'extra_scores': [],
        'flags':        ['flag_non_response', 'flag_omission_error_rate', 'flag_trial_flags'],
    },
    'delaydiscounting': {
        'score':        'lnk',
        'extra_scores': [],
        'flags':        ['flag_catch_trials', 'flag_median_rt'],
    },
    'flanker': {
        'score':        'rcs_interference',
        'extra_scores': [],
        'flags':        ['flag_accuracy', 'flag_trial_flags'],
    },
    'emorecog': {
        'score':        None,  # avg_accuracy derived from per-emotion accuracies
        'extra_scores': ['happy_accuracy', 'sad_accuracy', 'fearful_accuracy', 'angry_accuracy'],
        'flags':        ['flag_median_rtc', 'flag_same_response', 'flag_trial_flags'],
    },
}

# -- BQ-formatted concept lists -----------------------------------------------
MDD_BQ              = ','.join(str(c) for c in MDD_CONCEPTS)
DISQ_BQ             = ','.join(str(c) for c in DISQUALIFYING_CONCEPTS)
SUD_BQ              = ','.join(str(c) for c in SUD_EXCEPTIONS)
HC_ANC_BQ           = ','.join(str(c) for c in (MENTAL_DISORDER_ANCESTOR,) + HC_EXTRA_EXCLUSIONS)
BD_BQ               = ','.join(str(c) for c in BD_CONCEPTS)
PSYCHOSIS_BP_LEGACY_BQ = ','.join(str(c) for c in PSYCHOSIS_BP_LEGACY_IDS)

MDD_SET        = set(MDD_CONCEPTS)
DISQ_SET       = set(DISQUALIFYING_CONCEPTS)
BD_SET         = set(BD_CONCEPTS)
BD_EPISODE_SET = set(BD_EPISODE_CONCEPTS)
BD_DISQ_SET    = set(BD_DISQ_CONCEPTS)

RES_OUT = "nb01"

print(f'OVERWRITE                  : {OVERWRITE}')
print(f'N_WASH_DAYS                : {N_WASH_DAYS}')
print(f'N_DAYS_WEARABLE            : {N_DAYS_WEARABLE}')
print(f'EPISODE_GAP_DAYS           : {EPISODE_GAP_DAYS}')
print(f'EPISODE_PSYCHOSIS_GAP_DAYS : {EPISODE_PSYCHOSIS_GAP_DAYS}')
print(f'BD_DISQ_CONCEPTS           : {BD_DISQ_CONCEPTS}')
print(f'PSYCHOSIS_DEP_ANCESTOR     : {PSYCHOSIS_DEP_ANCESTOR}')
print(f'PSYCHOSIS_BP_ANCESTOR      : {PSYCHOSIS_BP_ANCESTOR}')
print(f'PSYCHOSIS_BP_LEGACY_IDS    : {PSYCHOSIS_BP_LEGACY_IDS}')

## Task 1: EtM Data

Scores and QC flags for all participants who completed any EtM task. Repeat administrations
are ordered by start time, then `sitting_id`, and only the first is kept.

### EtM QC flags and primary scores

A session is excluded if any of its QC flags is set (eTable 1).

| Task | Primary score | Direction | QC flag | Exclusion criterion |
|---|---|---|---|---|
| GradCPT | `dprime` | higher = better attention | `flag_non_response` | ≥75 consecutive non-responses (~60 s) |
| | | | `flag_omission_error_rate` | >50% omissions on go trials |
| | | | `flag_trial_flags` | cumulative flagged trial duration >60 s |
| Flanker | `rcs_interference` | higher = more interference = worse control | `flag_accuracy` | overall accuracy <0.65 |
| | | | `flag_trial_flags` | >10% of trials with RT <200 ms |
| Delay Discounting | `lnk` | higher = more impulsive | `flag_catch_trials` | catch trial accuracy <0.75 |
| | | | `flag_median_rt` | median RT <500 ms |
| Emotion Recognition | `avg_accuracy` (derived) | higher = better recognition | `flag_median_rtc` | median correct RT <700 ms |
| | | | `flag_same_response` | any single response used >90% of trials |
| | | | `flag_trial_flags` | >10% of trials with RT <300 ms |

`avg_accuracy` is the mean of the four per-emotion accuracies. `positive_bias` (happy minus
the mean of the other three) is kept for description only.

In [ ]:
# -- 1a: Pull EtM data per task (all administrations, no PID pre-filter) -----
def order_sittings(df):
    """Sort sittings by person, full start timestamp, then sitting_id (stable).

    Same-day sittings are ordered by time rather than by row order, and sitting_id
    breaks any remaining tie, so the first sitting does not depend on the pull.
    """
    return df.sort_values(['person_id', 'admin_ts', 'sitting_id'], kind='stable')

etm_raw = {}

for task in ETM_TASKS:
    cache_path   = f'raw_pulls/etm/{task}.parquet'
    schema       = ETM_SCHEMA[task]
    score_col    = schema['score']          # None for emorecog
    extra_scores = schema['extra_scores']
    flag_cols    = schema['flags']

    # Build SELECT clauses: primary score (if any) + extra scores + QC flags
    score_parts   = ([f'outcomes.{score_col} AS {score_col}'] if score_col else [])
    extra_parts   = [f'outcomes.{s} AS {s}' for s in extra_scores]
    all_score_parts = score_parts + extra_parts
    score_selects = ',\n            '.join(all_score_parts)
    flag_selects  = ',\n            '.join(
        f'outcomes.{f} AS {f}' for f in flag_cols
    )

    if not gcs_exists(cache_path) or OVERWRITE:
        score_block = f'{score_selects},\n            ' if score_selects else ''
        df = client.query(f'''
        SELECT
            sitting_id,
            person_id,
            {score_block}{flag_selects},
            metadata.touch AS touch,
            DATE(metadata.test_start_date_time) AS admin_date,
            metadata.test_start_date_time AS admin_ts
        FROM {tbl(task)}
        ''').to_dataframe()
        df['admin_date'] = pd.to_datetime(df['admin_date'])
        df['admin_ts']   = pd.to_datetime(df['admin_ts'], utc=True)
        save_parquet(df, cache_path)
        print(f'{task}: queried and cached {len(df):,} rows, '
              f'{df["person_id"].nunique():,} PIDs')
    else:
        df = load_parquet(cache_path)
        df['admin_date'] = pd.to_datetime(df['admin_date'])
        df['admin_ts']   = pd.to_datetime(df['admin_ts'], utc=True)
        print(f'{task}: loaded {len(df):,} rows, '
              f'{df["person_id"].nunique():,} PIDs')

    etm_raw[task] = df

In [ ]:
# -- 1b: Diagnostics ----------------------------------------------------------

# Participation per task
print('=== Participation per task ===')
pid_sets = {}
for task in ETM_TASKS:
    df   = etm_raw[task]
    pids = set(df['person_id'].unique())
    pid_sets[task] = pids
    n_multi = (df.groupby('person_id').size() > 1).sum()
    print(f'  {task:<20}: {len(pids):>6,} PIDs | {n_multi:>4,} with >1 administration')

any_etm_set  = set.union(*pid_sets.values())
all4_etm_set = set.intersection(*pid_sets.values())
print(f'\n  Any task   : {len(any_etm_set):,}')
print(f'  All 4 tasks: {len(all4_etm_set):,}')

task_count = Counter()
for pid in any_etm_set:
    task_count[sum(pid in pid_sets[t] for t in ETM_TASKS)] += 1
print('\n  Tasks completed (N participants):')
for n in sorted(task_count):
    print(f'    {n} task(s): {task_count[n]:,}')

# Administration count distribution per task
print('\n=== Administration count distribution per task ===')
fig, axes = plt.subplots(1, len(ETM_TASKS), figsize=(14, 3))
for ax, task in zip(axes, ETM_TASKS):
    counts = (
        etm_raw[task]
        .groupby('person_id')
        .size()
        .value_counts()
        .sort_index()
    )
    ax.bar(counts.index, counts.values, color='steelblue', edgecolor='white')
    ax.set_xlabel('Number of administrations')
    ax.set_ylabel('Participants')
    ax.set_title(task)
    ax.set_xticks(counts.index)
    for n, v in counts.items():
        ax.text(n, v + max(counts.values) * 0.01, str(v),
                ha='center', va='bottom', fontsize=8)
save_res(RES_OUT, 'admin_count_distribution.png', overwrite=True, res=fig)
plt.suptitle('Administration count per task', y=1.02)
plt.tight_layout()
plt.show()

# Repeat administration gap distribution
print('\n=== Inter-session gaps (repeat administrations) ===')
fig, axes = plt.subplots(1, len(ETM_TASKS), figsize=(14, 3))
for ax, task in zip(axes, ETM_TASKS):
    df = order_sittings(etm_raw[task])
    repeat_pids = df.groupby('person_id').filter(lambda x: len(x) > 1)
    if len(repeat_pids) == 0:
        ax.set_title(f'{task}\n(no repeats)')
        ax.axis('off')
        continue
    gaps = (
        repeat_pids.groupby('person_id')['admin_date']
        .apply(lambda x: x.diff().dt.days.dropna())
        .reset_index(drop=True)
    )
    ax.hist(gaps, bins=20, color='steelblue', edgecolor='white')
    ax.axvline(30, color='firebrick', ls='--', lw=1.2, label='30 d')
    ax.set_title(f'{task}\nN={len(gaps):,} gaps')
    ax.set_xlabel('Days between sessions')
    ax.legend(fontsize=7)
save_res(RES_OUT, "inter_session_gaps.png", overwrite=True, res=fig)
plt.suptitle('Inter-session gap — repeat EtM administrations', y=1.02)
plt.tight_layout()
plt.show()

# Multi-task date spread (all EtM completers)
print('=== Multi-task date spread (first admin per task) ===')
first_dates = {
    task: (
        order_sittings(etm_raw[task])
        .drop_duplicates('person_id', keep='first')
        [['person_id', 'admin_date']]
        .rename(columns={'admin_date': f'date_{task}'})
    )
    for task in ETM_TASKS
}
spread_df = reduce(
    lambda a, b: a.merge(b, on='person_id', how='outer'),
    first_dates.values()
)
_date_cols = [f'date_{t}' for t in ETM_TASKS]
spread_df['spread_days'] = (
    spread_df[_date_cols].max(axis=1) - spread_df[_date_cols].min(axis=1)
).dt.days

has_multi = spread_df.dropna(subset=['spread_days'])
print(f'PIDs with >= 2 tasks: {len(has_multi):,}')
print(has_multi['spread_days'].describe().round(1))

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(has_multi['spread_days'], bins=30, color='steelblue', edgecolor='white')
ax.axvline(30, color='firebrick', ls='--', lw=1.2, label='30 days')
ax.set_xlabel('Spread (days): max task date − min task date')
ax.set_ylabel('Participants')
ax.set_title('Date spread across EtM tasks (first admin per task)')
ax.legend()
save_res(RES_OUT, "date_spread_across_EtM_tasks.png", overwrite=True, res=fig)
plt.tight_layout()
plt.show()

In [ ]:
# -- 1c: Long-format parquet of all administrations for repeat-takers ---------
all_flag_cols = sorted({
    f
    for task in ETM_TASKS
    for f in ETM_SCHEMA[task]['flags']
})

# Derived from the cached pulls above, so rebuilt on every run (no OVERWRITE gate)
parts = []
for task in ETM_TASKS:
    df        = etm_raw[task].copy()
    schema    = ETM_SCHEMA[task]
    score_col = schema['score']   # None for emorecog
    flag_cols = schema['flags']

    df = order_sittings(df)
    df['admin_number'] = df.groupby('person_id').cumcount() + 1

    repeat_mask = df.groupby('person_id')['admin_number'].transform('max') > 1
    df = df[repeat_mask].copy()

    # score_raw is the task's score column; emorecog has none, so this descriptive
    # repeat-administration file stores positive_bias for it
    if score_col is not None:
        df = df.rename(columns={score_col: 'score_raw'})
    else:
        df['score_raw'] = (
            df['happy_accuracy']
            - (df['sad_accuracy'] + df['fearful_accuracy'] + df['angry_accuracy']) / 3
        )

    df['task']       = task
    df['qc_flagged'] = df[flag_cols].fillna(0).astype(int).max(axis=1) >= 1

    for fc in all_flag_cols:
        if fc not in df.columns:
            df[fc] = np.nan

    cols = ['person_id', 'task', 'sitting_id', 'admin_number', 'admin_date',
            'score_raw', 'qc_flagged'] + all_flag_cols
    parts.append(df[cols])

repeat_admins_df = (
    pd.concat(parts, ignore_index=True)
    .sort_values(['person_id', 'task', 'admin_number'])
    .reset_index(drop=True)
)
save_parquet(repeat_admins_df, 'raw_pulls/etm_repeat_admins.parquet')
print(f'Saved etm_repeat_admins: {len(repeat_admins_df):,} rows | '
      f'{repeat_admins_df["person_id"].nunique():,} unique PIDs')

print('\nRepeat-takers per task:')
print(
    repeat_admins_df.groupby('task')['person_id']
    .nunique()
    .rename('repeat_takers')
    .to_frame()
)

In [ ]:
# -- 1d: First administration per task; per-task QC flags; etm_pids ----------

# First administration per person per task (no cross-task QC exclusion)
etm_first = {
    task: (
        order_sittings(etm_raw[task])
        .drop_duplicates('person_id', keep='first')
        .copy()
    )
    for task in ETM_TASKS
}

# Per-task qc_flagged column — stored as data; exclusion is applied per task in the
# modelling notebooks (04a–04c)
def any_flag_set(df, flag_cols):
    return df[flag_cols].fillna(0).astype(int).max(axis=1) >= 1

print('=== EtM QC flags (per task, stored as column; no cross-task exclusion) ===')
for task in ETM_TASKS:
    df        = etm_first[task]
    flag_cols = ETM_SCHEMA[task]['flags']
    df['qc_flagged'] = any_flag_set(df, flag_cols)
    n_flagged = int(df['qc_flagged'].sum())
    print(f'  {task:<20}: {df["person_id"].nunique():,} PIDs | '
          f'{n_flagged:,} flagged ({100*n_flagged/len(df):.1f}%)')
    etm_first[task] = df

# Guard: an all-null flag column would silently pass every emorecog session
print('\n=== emorecog QC flag sanity check ===')
emorecog_df = etm_first['emorecog']
all_ok = True
for fc in ETM_SCHEMA['emorecog']['flags']:
    n_null  = int(emorecog_df[fc].isna().sum())
    n_total = len(emorecog_df)
    vc      = emorecog_df[fc].value_counts(dropna=False).to_dict()
    status  = 'ALL NULL' if n_null == n_total else 'ok'
    print(f'  {fc}: {n_null}/{n_total} null | values: {vc}  [{status}]')
    if n_null == n_total:
        all_ok = False
if not all_ok:
    raise ValueError(
        'emorecog: one or more flag columns are entirely null — '
        'QC is silently passing all rows. Check ETM_SCHEMA flag names.'
    )
else:
    print('  -> all emorecog flag columns contain non-null values.')

# Build etm_dates: per-task first-admin date + etm_date_first (min across tasks)
etm_dates = reduce(
    lambda a, b: a.merge(b, on='person_id', how='outer'),
    [
        etm_first[t][['person_id', 'admin_date']]
        .rename(columns={'admin_date': f'date_{t}'})
        for t in ETM_TASKS
    ]
)
date_cols = [f'date_{t}' for t in ETM_TASKS]
etm_dates['etm_date_first'] = etm_dates[date_cols].min(axis=1)

# All EtM completers (any task) — base population for all downstream queries
etm_pids    = etm_dates['person_id'].tolist()
etm_pids_bq = ','.join(str(p) for p in etm_pids)

log_attrition(
    step     = '1. Any EtM task completed',
    group    = 'All',
    n_before = len(etm_pids),
    n_after  = len(etm_pids),
    reason   = 'Any of the 4 EtM tasks completed (full AoU pull, no pre-filter)',
)

print(f'\nTotal EtM completers (any task)  : {len(etm_pids):,}')
print(f'Completed all 4 tasks            : {etm_dates[date_cols].notna().all(axis=1).sum():,}')
tasks_completed = etm_dates[date_cols].notna().sum(axis=1).value_counts().sort_index()
print('\nTask count distribution:')
for n, cnt in tasks_completed.items():
    print(f'  {n} task(s): {cnt:,}')

## Task 2: Psychiatric Diagnoses

EHR records of MDD, BD and the excluding conditions for all EtM completers, at any date; the
test-date rules are applied in Tasks 3 and 4.

In [ ]:
if not gcs_exists('raw_pulls/psych_mdd_disq.parquet') or OVERWRITE:
    psych_df = client.query(f'''
    SELECT DISTINCT
        co.person_id,
        ca.ancestor_concept_id AS matched_concept_id,
        co.condition_start_date
    FROM {tbl('condition_occurrence')} co
    JOIN {tbl('concept_ancestor')} ca
        ON ca.descendant_concept_id = co.condition_concept_id
    WHERE ca.ancestor_concept_id IN ({MDD_BQ}, {DISQ_BQ})
      AND co.person_id IN ({etm_pids_bq})
      AND NOT (
          ca.ancestor_concept_id = 440069
          AND co.condition_concept_id IN (
              SELECT descendant_concept_id
              FROM {tbl('concept_ancestor')}
              WHERE ancestor_concept_id IN ({SUD_BQ})
          )
      )
    ''').to_dataframe()
    psych_df['condition_start_date'] = pd.to_datetime(psych_df['condition_start_date'])
    save_parquet(psych_df, 'raw_pulls/psych_mdd_disq.parquet')
    print(f'Queried and cached: {len(psych_df):,} records, '
          f'{psych_df["person_id"].nunique():,} persons')
else:
    psych_df = load_parquet('raw_pulls/psych_mdd_disq.parquet')
    psych_df['condition_start_date'] = pd.to_datetime(psych_df['condition_start_date'])
    print(f'Loaded from cache: {len(psych_df):,} records, '
          f'{psych_df["person_id"].nunique():,} persons')

CONCEPT_LABELS = {
    4282096: 'MDD single episode',
    4282316: 'MDD recurrent',
    436665:  'Bipolar disorder',
    443237:  'Manic episode',
    435783:  'Schizophrenia',
    4286201: 'Schizoaffective disorder',
    438409:  'ADHD',
    439776:  'ASD',
    440069:  'Drug dependence',
    4182210: 'Dementia',
    443432:  'Cognitive impairment',
}

print('\nPersons per matched concept:')
for cid, label in CONCEPT_LABELS.items():
    n = psych_df.loc[
        psych_df['matched_concept_id'] == cid, 'person_id'
    ].nunique()
    if n > 0:
        print(f'  {label} ({cid}): {n:,}')

### Task 2b: NCC Exclusions

Any record under the mental-disorder ancestor (432586), plus drug dependence and cognitive
impairment, which are not under that ancestor in this CDR.

In [ ]:
if not gcs_exists('raw_pulls/psych_any_mental.parquet') or OVERWRITE:
    any_mental_df = client.query(f'''
    SELECT DISTINCT co.person_id
    FROM {tbl('condition_occurrence')} co
    JOIN {tbl('concept_ancestor')} ca
        ON ca.descendant_concept_id = co.condition_concept_id
    WHERE ca.ancestor_concept_id IN ({HC_ANC_BQ})
      AND co.person_id IN ({etm_pids_bq})
      AND NOT (
          ca.ancestor_concept_id = 440069
          AND co.condition_concept_id IN (
              SELECT descendant_concept_id
              FROM {tbl('concept_ancestor')}
              WHERE ancestor_concept_id IN ({SUD_BQ})
          )
      )
    ''').to_dataframe()
    save_parquet(any_mental_df, 'raw_pulls/psych_any_mental.parquet')
    print(f'Queried and cached: {len(any_mental_df):,} persons with any exclusionary condition')
else:
    any_mental_df = load_parquet('raw_pulls/psych_any_mental.parquet')
    print(f'Loaded from cache: {len(any_mental_df):,} persons with any exclusionary condition')

any_mental_pids = set(any_mental_df['person_id'])
pct = 100 * len(any_mental_pids) / len(etm_pids)
print(f'({pct:.1f}% of all EtM completers)')

## Task 3: Candidate Sets

- **MDD (per task):** ≥1 MDD record with `condition_start_date ≤ etm_date_{task}`; no
  `DISQUALIFYING_CONCEPTS` record at any date. That list includes the BD codes, so MDD and BD
  do not overlap.
- **BD (per task):** ≥1 BD record with `condition_start_date ≤ etm_date_{task}`; no
  `BD_DISQ_CONCEPTS` record at any date (MDD records allowed).
- **HC:** no record under `MENTAL_DISORDER_ANCESTOR` or `HC_EXTRA_EXCLUSIONS` at any date.

Nicotine and tobacco dependence (`SUD_EXCEPTIONS`) do not count as drug dependence. A
participant's `group` is MDD or BD if they qualify for at least one task.

In [ ]:
# -- Disqualifying exclusion (global — any EHR date) --------------------------
disq_pids_set = set(
    psych_df.loc[psych_df['matched_concept_id'].isin(DISQ_SET), 'person_id'].unique()
)

# -- Per-task MDD qualifying sets ---------------------------------------------
# ≥1 MDD record on or before the task date; disqualifying conditions at any date.
_mdd_all = psych_df[psych_df['matched_concept_id'].isin(MDD_SET)].copy()
mdd_any  = set(_mdd_all['person_id'].unique())

mdd_eligible_per_task = {}
for task in ETM_TASKS:
    date_col     = f'date_{task}'
    task_dates_k = etm_dates[['person_id', date_col]].dropna(subset=[date_col])
    task_joined  = _mdd_all.merge(task_dates_k, on='person_id', how='inner')
    task_pre     = task_joined[task_joined['condition_start_date'] <= task_joined[date_col]]
    eligible     = set(task_pre['person_id'].unique()) - disq_pids_set
    mdd_eligible_per_task[task] = eligible
    log_attrition(
        step     = f'3a. MDD qualifying — {task}',
        group    = 'MDD',
        n_before = len(etm_pids),
        n_after  = len(eligible),
        reason   = (f'>= 1 MDD record with condition_start_date <= etm_date_{task}; '
                    f'no DISQUALIFYING_CONCEPTS record (any EHR date)'),
        task     = task,
    )

# Union: MDD-eligible for at least one task
mdd_candidate_pids = set.union(*mdd_eligible_per_task.values())

# -- Diagnostic ---------------------------------------------------------------
mdd_not_any_task = mdd_any - disq_pids_set - mdd_candidate_pids
print(f'MDD records in EHR (no disqualifying condition) '
      f'but MDD onset after all completed task dates: {len(mdd_not_any_task):,}')
print()
for task in ETM_TASKS:
    print(f'  MDD eligible for {task:<20}: {len(mdd_eligible_per_task[task]):,}')
print(f'  MDD eligible (union, any task)  : {len(mdd_candidate_pids):,}')

# -- HC candidate set ---------------------------------------------------------
hc_candidate_pids = set(etm_pids) - any_mental_pids

log_attrition(
    step     = '3b. HC qualification',
    group    = 'HC',
    n_before = len(etm_pids),
    n_after  = len(hc_candidate_pids),
    reason   = 'No record under MENTAL_DISORDER_ANCESTOR or HC_EXTRA_EXCLUSIONS (any EHR date)',
)

# -- BD disqualifying exclusion -----------------------------------------------
# BD_DISQ_CONCEPTS = DISQUALIFYING minus BD codes; MDD history is allowed in BD participants
bd_disq_pids_set = set(
    psych_df.loc[psych_df['matched_concept_id'].isin(BD_DISQ_SET), 'person_id'].unique()
)

# -- Per-task BD qualifying sets ----------------------------------------------
# ≥1 BD record on or before the task date (BD codes are already in psych_df).
_bd_all = psych_df[psych_df['matched_concept_id'].isin(BD_SET)].copy()
bd_any  = set(_bd_all['person_id'].unique())

bd_eligible_per_task = {}
for task in ETM_TASKS:
    date_col     = f'date_{task}'
    task_dates_k = etm_dates[['person_id', date_col]].dropna(subset=[date_col])
    task_joined  = _bd_all.merge(task_dates_k, on='person_id', how='inner')
    task_pre     = task_joined[task_joined['condition_start_date'] <= task_joined[date_col]]
    eligible     = set(task_pre['person_id'].unique()) - bd_disq_pids_set
    bd_eligible_per_task[task] = eligible
    log_attrition(
        step     = f'3c. BD qualifying — {task}',
        group    = 'BD',
        n_before = len(etm_pids),
        n_after  = len(eligible),
        reason   = (f'>= 1 BD record with condition_start_date <= etm_date_{task}; '
                    f'no BD_DISQ_CONCEPTS record (any EHR date)'),
        task     = task,
    )

bd_candidate_pids = set.union(*bd_eligible_per_task.values())

# -- Summary ------------------------------------------------------------------
n_overlap_mdd_hc = len(mdd_candidate_pids & hc_candidate_pids)
n_overlap_mdd_bd = len(mdd_candidate_pids & bd_candidate_pids)
n_overlap_bd_hc  = len(bd_candidate_pids & hc_candidate_pids)
n_neither        = len(set(etm_pids) - mdd_candidate_pids - bd_candidate_pids - hc_candidate_pids)

print(f'\nMDD candidates (union, any task) : {len(mdd_candidate_pids):,}')
print(f'BD candidates (union, any task)  : {len(bd_candidate_pids):,}')
print(f'HC candidates                    : {len(hc_candidate_pids):,}')
print(f'MDD/HC overlap                   : {n_overlap_mdd_hc:,}  (expected 0)')
print(f'MDD/BD overlap                   : {n_overlap_mdd_bd:,}  (expected 0 — BD disqualifies MDD)')
print(f'BD/HC overlap                    : {n_overlap_bd_hc:,}   (expected 0)')
print(f'Neither                          : {n_neither:,}')

print(f'\nBD per-task eligibility:')
for task in ETM_TASKS:
    print(f'  BD eligible for {task:<20}: {len(bd_eligible_per_task[task]):,}')

In [ ]:
# -- Attrition cascade so far -------------------------------------------------
print('=== Attrition cascade (Tasks 1-3) ===\n')
attrition_df = pd.DataFrame(attrition_rows)
display(attrition_df[['step', 'group', 'n_before', 'n_dropped', 'n_after', 'reason']])

# -- MDD span diagnostic (informational) --------------------------------------
print('\n=== MDD candidate: EtM date spread across tasks ===')
spread_mdd = etm_dates[etm_dates['person_id'].isin(mdd_candidate_pids)].copy()
spread_mdd['date_first'] = spread_mdd[date_cols].min(axis=1)
spread_mdd['date_last']  = spread_mdd[date_cols].max(axis=1)
spread_mdd['span_days']  = (spread_mdd['date_last'] - spread_mdd['date_first']).dt.days

n_any     = len(spread_mdd)
same_day  = (spread_mdd['span_days'] == 0).sum()
print(f'MDD candidates : {n_any:,}')
print(f'All tasks on same day: {same_day:,} ({100*same_day/n_any:.1f}%)')
print('\nSpan distribution (days, MDD candidates):')
print(spread_mdd['span_days'].describe(percentiles=[.5, .75, .9, .95]).round(1).to_string())

## Task 4: Washout and Fitbit Eligibility

Per task, for MDD and BD:
- `etm_washout_pass_{task}`: no mood-disorder code within 90 days of the test, in either direction
- `fitbit_sufficient_{task}`: Fitbit recording started at least 90 days before the test
- `wearable_washout_pass_{task}`: no mood-disorder code in the 90 days before the wearable
  window, `[etm_date − 180 d, etm_date − 90 d)`

MDD uses MDD codes; BD uses MDD and BD codes. HC get `fitbit_sufficient_{task}` only (Task 6).

In [ ]:
# -- 4a: Load Fitbit coverage for MDD and BD candidates ----------------------
# Per-participant first/last Fitbit date, written by 00_build_cache to the cache bucket.
coverage_df = load_cache_parquet('meta/fitbit_coverage.parquet')
coverage_df['fitbit_start'] = pd.to_datetime(coverage_df['fitbit_start'])
coverage_df['fitbit_end']   = pd.to_datetime(coverage_df['fitbit_end'])

fitbit_mdd = coverage_df[coverage_df['person_id'].isin(mdd_candidate_pids)].copy()
fitbit_bd  = coverage_df[coverage_df['person_id'].isin(bd_candidate_pids)].copy()
print(f'MDD candidates with Fitbit data : {len(fitbit_mdd):,} / {len(mdd_candidate_pids):,}')
print(f'BD candidates with Fitbit data  : {len(fitbit_bd):,} / {len(bd_candidate_pids):,}')

# -- 4b: Per-task eligibility flags ------------------------------------------
n_wash = pd.Timedelta(days=N_WASH_DAYS)

# MDD washout uses MDD codes only; BD washout uses MDD + BD codes (bipolar can manifest as either polarity)
mdd_records_all = (
    psych_df[psych_df['matched_concept_id'].isin(MDD_SET)]
    [['person_id', 'condition_start_date']]
    .copy()
)
bd_records_all = (
    psych_df[psych_df['matched_concept_id'].isin(BD_EPISODE_SET)]
    [['person_id', 'condition_start_date']]
    .copy()
)

def compute_per_task_flags(eligible_per_task, episode_records, fitbit_coverage, group_label):
    """Compute etm_washout_pass, fitbit_sufficient, wearable_washout_pass per task."""
    frame_list = []
    for task in ETM_TASKS:
        task_dates = (
            etm_dates[['person_id', f'date_{task}']]
            .rename(columns={f'date_{task}': 'task_date'})
        )
        task_df = (
            pd.DataFrame({'person_id': sorted(eligible_per_task[task])})
            .merge(task_dates, on='person_id', how='left')
            .merge(fitbit_coverage[['person_id', 'fitbit_start', 'fitbit_end']], on='person_id', how='left')
        )

        has_task   = task_df['task_date'].notna()
        has_fitbit = task_df['fitbit_start'].notna()

        fit_suf = has_task & has_fitbit & (
            task_df['fitbit_start'] <= task_df['task_date'] - pd.Timedelta(days=N_DAYS_WEARABLE)
        )
        task_df[f'fitbit_sufficient_{task}'] = np.where(has_task, fit_suf.astype(float), np.nan)

        completed = task_df[has_task][['person_id', 'task_date']]
        joined    = completed.merge(episode_records, on='person_id', how='inner')
        d         = joined['condition_start_date']

        joined['in_etm_washout']  = (
            (d >= joined['task_date'] - n_wash) & (d <= joined['task_date'] + n_wash)
        )
        joined['in_wear_washout'] = (
            (d >= joined['task_date'] - 2 * n_wash) & (d < joined['task_date'] - n_wash)
        )

        fail_etm  = set(joined.loc[joined['in_etm_washout'],  'person_id'])
        fail_wear = set(joined.loc[joined['in_wear_washout'], 'person_id'])

        task_df[f'etm_washout_pass_{task}'] = np.where(
            ~has_task, np.nan,
            (~task_df['person_id'].isin(fail_etm)).astype(float)
        )
        task_df[f'wearable_washout_pass_{task}'] = np.where(
            ~has_task, np.nan,
            (~task_df['person_id'].isin(fail_wear)).astype(float)
        )

        n_task      = int(has_task.sum())
        n_etm_pass  = int((has_task & ~task_df['person_id'].isin(fail_etm)).sum())
        n_fit_pass  = int(fit_suf.sum())
        n_wear_pass = int((has_task & ~task_df['person_id'].isin(fail_wear)).sum())

        log_attrition(
            step     = f'4a. EtM washout — {task} ({group_label})',
            group    = group_label,
            n_before = n_task,
            n_after  = n_etm_pass,
            reason   = f'No episode EHR record within ±{N_WASH_DAYS}d of etm_date_{task}',
            task     = task,
        )
        log_attrition(
            step     = f'4b. Fitbit sufficient — {task} ({group_label})',
            group    = group_label,
            n_before = n_task,
            n_after  = n_fit_pass,
            reason   = f'fitbit_start <= etm_date_{task} - {N_DAYS_WEARABLE}d',
            task     = task,
        )
        log_attrition(
            step     = f'4c. Wearable washout — {task} ({group_label})',
            group    = group_label,
            n_before = n_task,
            n_after  = n_wear_pass,
            reason   = f'No episode EHR record in [etm_date_{task} - {2*N_WASH_DAYS}d, etm_date_{task} - {N_WASH_DAYS}d]',
            task     = task,
        )

        flag_cols_task = [
            f'etm_washout_pass_{task}',
            f'fitbit_sufficient_{task}',
            f'wearable_washout_pass_{task}',
        ]
        frame_list.append(task_df[['person_id'] + flag_cols_task])

    return reduce(lambda a, b: a.merge(b, on='person_id', how='outer'), frame_list)


mdd_flags_df = compute_per_task_flags(mdd_eligible_per_task, mdd_records_all, fitbit_mdd, 'MDD')
bd_flags_df  = compute_per_task_flags(bd_eligible_per_task,  bd_records_all,  fitbit_bd,  'BD')

# MDD and BD are disjoint so no person_id conflicts
flags_df = pd.concat([mdd_flags_df, bd_flags_df], ignore_index=True)

print('\nPer-task flag summary:')
for group_label, elig_per_task in [('MDD', mdd_eligible_per_task), ('BD', bd_eligible_per_task)]:
    print(f'\n  {group_label}:')
    grp_pids = set().union(*elig_per_task.values())
    grp_flags = flags_df[flags_df['person_id'].isin(grp_pids)]
    for task in ETM_TASKS:
        n_task = len(elig_per_task[task])
        task_pids = elig_per_task[task]
        sub = grp_flags[grp_flags['person_id'].isin(task_pids)]
        etm_p  = int(sub[f'etm_washout_pass_{task}'].fillna(0).sum())
        fit_p  = int(sub[f'fitbit_sufficient_{task}'].fillna(0).sum())
        wear_p = int(sub[f'wearable_washout_pass_{task}'].fillna(0).sum())
        print(f'    {task}: {n_task:,} eligible | '
              f'EtM washout pass: {etm_p:,} | Fitbit sufficient: {fit_p:,} | '
              f'Wearable washout: {wear_p:,}')

## Task 5: Prior Episodes (descriptive)

Number of mood episodes before each test, counting records within 180 days of each other as one
episode (MDD codes for MDD; MDD and BD codes for BD). Prior psychotic episodes are counted the
same way, with a 60-day gap. Not used in any model.

In [ ]:
# -- 5a: Pull psychosis episode records (MDD and BD candidates only) ----------
clinical_candidate_pids = sorted(mdd_candidate_pids | bd_candidate_pids)
clinical_cands_bq       = ','.join(str(p) for p in clinical_candidate_pids)

if not gcs_exists('raw_pulls/psych_psychosis_episodes.parquet') or OVERWRITE:
    psychosis_df = client.query(f'''
    SELECT DISTINCT co.person_id, 'dep_psychosis' AS psychosis_type, co.condition_start_date
    FROM {tbl('condition_occurrence')} co
    JOIN {tbl('concept_ancestor')} ca
        ON ca.descendant_concept_id = co.condition_concept_id
        AND ca.ancestor_concept_id = {PSYCHOSIS_DEP_ANCESTOR}
    WHERE co.person_id IN ({clinical_cands_bq})

    UNION DISTINCT

    SELECT DISTINCT co.person_id, 'bp_psychosis' AS psychosis_type, co.condition_start_date
    FROM {tbl('condition_occurrence')} co
    JOIN {tbl('concept_ancestor')} ca
        ON ca.descendant_concept_id = co.condition_concept_id
        AND ca.ancestor_concept_id = {PSYCHOSIS_BP_ANCESTOR}
    WHERE co.person_id IN ({clinical_cands_bq})

    UNION DISTINCT

    SELECT DISTINCT co.person_id, 'bp_psychosis' AS psychosis_type, co.condition_start_date
    FROM {tbl('condition_occurrence')} co
    WHERE co.condition_concept_id IN ({PSYCHOSIS_BP_LEGACY_BQ})
      AND co.person_id IN ({clinical_cands_bq})
    ''').to_dataframe()
    psychosis_df['condition_start_date'] = pd.to_datetime(psychosis_df['condition_start_date'])
    save_parquet(psychosis_df, 'raw_pulls/psych_psychosis_episodes.parquet')
    print(f'Queried and cached: {len(psychosis_df):,} records, '
          f'{psychosis_df["person_id"].nunique():,} persons')
else:
    psychosis_df = load_parquet('raw_pulls/psych_psychosis_episodes.parquet')
    psychosis_df['condition_start_date'] = pd.to_datetime(psychosis_df['condition_start_date'])
    print(f'Loaded from cache: {len(psychosis_df):,} records, '
          f'{psychosis_df["person_id"].nunique():,} persons')

print('\nRecords by psychosis type:')
print(
    psychosis_df.groupby('psychosis_type')['person_id']
    .agg(['count', 'nunique'])
    .rename(columns={'count': 'records', 'nunique': 'persons'})
)

# Assertion: all bp_psychosis concepts live under 436665 (disqualifying for MDD),
# so no MDD candidate should have bp_psychosis records
mdd_with_bp = set(
    psychosis_df.loc[
        (psychosis_df['psychosis_type'] == 'bp_psychosis') &
        (psychosis_df['person_id'].isin(mdd_candidate_pids)),
        'person_id'
    ]
)
assert len(mdd_with_bp) == 0, (
    f'Assertion failed: {len(mdd_with_bp):,} MDD candidates have bp_psychosis records.'
)
print('\nAssertion passed: no MDD candidate has bipolar psychosis records.')

In [ ]:
def collapse_episodes(dates, gap_days=EPISODE_GAP_DAYS):
    """Collapse mood occurrence dates into distinct episode onsets."""
    if not dates:
        return []
    sorted_dates = sorted(set(dates))
    episode_starts = [sorted_dates[0]]
    last_date = sorted_dates[0]
    for d in sorted_dates[1:]:
        if (d - last_date).days > gap_days:
            episode_starts.append(d)
        last_date = d
    return episode_starts

task_date_maps = {
    task: etm_dates.set_index('person_id')[f'date_{task}'].to_dict()
    for task in ETM_TASKS
}

# -- 5b: General episode history (gap = EPISODE_GAP_DAYS = 180d) --------------
mdd_records_all_ep = (
    psych_df[psych_df['matched_concept_id'].isin(MDD_SET)]
    [['person_id', 'condition_start_date']]
    .copy()
)
mdd_dates_by_pid = (
    mdd_records_all_ep
    .groupby('person_id')['condition_start_date']
    .apply(list)
    .to_dict()
)

episode_rows = []
for pid in tqdm(sorted(mdd_candidate_pids), desc='Episode history MDD (per task)'):
    all_dates = mdd_dates_by_pid.get(pid, [])
    row = {'person_id': pid}
    for task in ETM_TASKS:
        task_date = task_date_maps[task].get(pid, pd.NaT)
        if pd.isna(task_date) or pid not in mdd_eligible_per_task[task]:
            row[f'n_prior_episodes_{task}'] = np.nan
        else:
            pre_task = [d for d in all_dates if d < task_date]
            row[f'n_prior_episodes_{task}'] = len(collapse_episodes(pre_task))
    episode_rows.append(row)

episode_df = pd.DataFrame(episode_rows)

bd_records_all_ep = (
    psych_df[psych_df['matched_concept_id'].isin(BD_EPISODE_SET)]
    [['person_id', 'condition_start_date']]
    .copy()
)
bd_dates_by_pid = (
    bd_records_all_ep
    .groupby('person_id')['condition_start_date']
    .apply(list)
    .to_dict()
)

bd_episode_rows = []
for pid in tqdm(sorted(bd_candidate_pids), desc='Episode history BD (per task)'):
    all_dates = bd_dates_by_pid.get(pid, [])
    row = {'person_id': pid}
    for task in ETM_TASKS:
        task_date = task_date_maps[task].get(pid, pd.NaT)
        if pd.isna(task_date) or pid not in bd_eligible_per_task[task]:
            row[f'n_prior_episodes_{task}'] = np.nan
        else:
            pre_task = [d for d in all_dates if d < task_date]
            row[f'n_prior_episodes_{task}'] = len(collapse_episodes(pre_task))
    bd_episode_rows.append(row)

bd_episode_df = pd.DataFrame(bd_episode_rows)

# -- 5c: Psychosis episode history (gap = EPISODE_PSYCHOSIS_GAP_DAYS = 60d) --
dep_psychosis_dates_by_pid = (
    psychosis_df[psychosis_df['psychosis_type'] == 'dep_psychosis']
    .groupby('person_id')['condition_start_date']
    .apply(list)
    .to_dict()
)
bp_psychosis_dates_by_pid = (
    psychosis_df[psychosis_df['psychosis_type'] == 'bp_psychosis']
    .groupby('person_id')['condition_start_date']
    .apply(list)
    .to_dict()
)

# MDD: depressive psychosis episodes (unipolar tree, ancestor 37111697)
mdd_psychosis_rows = []
for pid in tqdm(sorted(mdd_candidate_pids), desc='Psychosis episode history MDD'):
    all_dates = dep_psychosis_dates_by_pid.get(pid, [])
    row = {'person_id': pid}
    for task in ETM_TASKS:
        task_date = task_date_maps[task].get(pid, pd.NaT)
        if pd.isna(task_date) or pid not in mdd_eligible_per_task[task]:
            row[f'n_prior_dep_psychosis_eps_{task}'] = np.nan
        else:
            pre_task = [d for d in all_dates if d < task_date]
            row[f'n_prior_dep_psychosis_eps_{task}'] = len(
                collapse_episodes(pre_task, gap_days=EPISODE_PSYCHOSIS_GAP_DAYS)
            )
    mdd_psychosis_rows.append(row)

mdd_psychosis_df = pd.DataFrame(mdd_psychosis_rows)

# BD: bipolar psychosis episodes (ancestor 4195158 + legacy IDs)
bd_psychosis_rows = []
for pid in tqdm(sorted(bd_candidate_pids), desc='Psychosis episode history BD'):
    all_dates = bp_psychosis_dates_by_pid.get(pid, [])
    row = {'person_id': pid}
    for task in ETM_TASKS:
        task_date = task_date_maps[task].get(pid, pd.NaT)
        if pd.isna(task_date) or pid not in bd_eligible_per_task[task]:
            row[f'n_prior_bp_psychosis_eps_{task}'] = np.nan
        else:
            pre_task = [d for d in all_dates if d < task_date]
            row[f'n_prior_bp_psychosis_eps_{task}'] = len(
                collapse_episodes(pre_task, gap_days=EPISODE_PSYCHOSIS_GAP_DAYS)
            )
    bd_psychosis_rows.append(row)

bd_psychosis_df = pd.DataFrame(bd_psychosis_rows)

# Merge psychosis columns into general episode dataframes before combining
episode_df    = episode_df.merge(mdd_psychosis_df, on='person_id', how='left')
bd_episode_df = bd_episode_df.merge(bd_psychosis_df, on='person_id', how='left')

# Combine for cohort_master assembly (MDD and BD are disjoint)
all_episode_df = pd.concat([episode_df, bd_episode_df], ignore_index=True)

# -- Summary ------------------------------------------------------------------
print(f'Episode history computed for {len(episode_df):,} MDD and {len(bd_episode_df):,} BD candidates\n')

print('General episodes (gap=180d):')
for group_label, ep_df, elig_per_task in [
    ('MDD', episode_df,    mdd_eligible_per_task),
    ('BD',  bd_episode_df, bd_eligible_per_task),
]:
    print(f'\n  {group_label}:')
    for task in ETM_TASKS:
        col     = f'n_prior_episodes_{task}'
        n_valid = int(ep_df[col].notna().sum())
        mean_ep = ep_df[col].mean()
        print(f'    {task}: {n_valid:,} eligible | mean = {mean_ep:.1f}')
    for task in ETM_TASKS:
        col    = f'n_prior_episodes_{task}'
        n_zero = int((ep_df[col] == 0).sum())
        if n_zero > 0:
            print(f'    WARNING ({group_label}): {n_zero:,} participants have 0 prior episodes before {task}.')
        else:
            print(f'    QC pass ({group_label}): no 0-episode participants before {task}.')

print('\nPsychosis episodes (gap=60d):')
print('\n  MDD — depressive psychosis (ancestor 37111697):')
for task in ETM_TASKS:
    col     = f'n_prior_dep_psychosis_eps_{task}'
    n_valid = int(episode_df[col].notna().sum())
    n_any   = int((episode_df[col] > 0).sum())
    mean_ep = episode_df.loc[episode_df[col].notna(), col].mean()
    print(f'    {task}: {n_valid:,} eligible | {n_any:,} with ≥1 | mean = {mean_ep:.2f}')

print('\n  BD — bipolar psychosis (ancestor 4195158 + legacy IDs):')
for task in ETM_TASKS:
    col     = f'n_prior_bp_psychosis_eps_{task}'
    n_valid = int(bd_episode_df[col].notna().sum())
    n_any   = int((bd_episode_df[col] > 0).sum())
    mean_ep = bd_episode_df.loc[bd_episode_df[col].notna(), col].mean()
    print(f'    {task}: {n_valid:,} eligible | {n_any:,} with ≥1 | mean = {mean_ep:.2f}')

## Task 6: Assemble and Save cohort_master

One row per participant: group, per-task eligibility, washout and Fitbit flags, test dates,
scores, QC flags, device type, prior episodes and Fitbit coverage. Task columns carry the task
name (e.g. `gradcpt_dprime`); a field that does not apply is NaN.

In [ ]:
# -- 6a: Score and flag column rename maps ------------------------------------
SCORE_RENAME = {
    'gradcpt': {
        'dprime': 'gradcpt_dprime',
    },
    'delaydiscounting': {
        'lnk': 'delaydiscounting_lnk',
    },
    'flanker': {
        'rcs_interference': 'flanker_rcs_interference',
    },
    'emorecog': {
        'happy_accuracy':   'emorecog_happy_accuracy',
        'sad_accuracy':     'emorecog_sad_accuracy',
        'fearful_accuracy': 'emorecog_fearful_accuracy',
        'angry_accuracy':   'emorecog_angry_accuracy',
    },
}
FLAG_PREFIX = {
    'gradcpt':          'gradcpt',
    'delaydiscounting': 'delaydiscounting',
    'flanker':          'flanker',
    'emorecog':         'emorecog',
}

# -- 6b: Build base frame: all candidates (MDD + BD + HC) --------------------
mdd_cands = pd.DataFrame({'person_id': sorted(mdd_candidate_pids), 'group': 'MDD'})
bd_cands  = pd.DataFrame({'person_id': sorted(bd_candidate_pids),  'group': 'BD'})
hc_cands  = pd.DataFrame({'person_id': sorted(hc_candidate_pids),  'group': 'HC'})
cohort    = pd.concat([mdd_cands, bd_cands, hc_cands], ignore_index=True)

# -- 6c: Merge EtM dates ------------------------------------------------------
etm_date_cols = ['person_id', 'etm_date_first'] + [f'date_{t}' for t in ETM_TASKS]
cohort = cohort.merge(etm_dates[etm_date_cols], on='person_id', how='left')
for task in ETM_TASKS:
    cohort = cohort.rename(columns={f'date_{task}': f'etm_date_{task}'})

# -- 6d: Merge Fitbit coverage (all groups; HC populated where in cache) ------
# coverage_df comes from the AoU-wide cache meta — HC with Fitbit data are included.
cohort = cohort.merge(
    coverage_df[['person_id', 'fitbit_start', 'fitbit_end']],
    on='person_id', how='left'
)

# -- 6e: Merge EtM scores and QC flags (one left join per task) ---------------
for task in ETM_TASKS:
    schema       = ETM_SCHEMA[task]
    score_col    = schema['score']        # None for emorecog
    extra_scores = schema['extra_scores']
    flag_cols    = schema['flags']
    prefix       = FLAG_PREFIX[task]

    score_part = [score_col] if score_col else []
    cols   = ['person_id'] + score_part + extra_scores + flag_cols + ['touch']
    task_df = etm_first[task][cols].copy()

    task_df = task_df.rename(columns=SCORE_RENAME[task])
    task_df = task_df.rename(columns={f: f'{prefix}_{f}' for f in flag_cols})
    task_df = task_df.rename(columns={'touch': f'touch_{task}'})

    cohort = cohort.merge(task_df, on='person_id', how='left')

# -- 6f: Derive emorecog_positive_bias ----------------------------------------
cohort['emorecog_positive_bias'] = (
    cohort['emorecog_happy_accuracy']
    - (  cohort['emorecog_sad_accuracy']
       + cohort['emorecog_fearful_accuracy']
       + cohort['emorecog_angry_accuracy']) / 3
)

# -- 6f2: Derive emorecog_avg_accuracy -----------------------------------------
# EmoRecog's model outcome. emorecog_positive_bias above is a descriptive column only.
cohort['emorecog_avg_accuracy'] = cohort[[
    'emorecog_happy_accuracy', 'emorecog_sad_accuracy',
    'emorecog_fearful_accuracy', 'emorecog_angry_accuracy',
]].mean(axis=1)

# -- 6g: Per-task group eligibility flags (group-specific; NaN for other groups) --
# 1.0 = eligible for this task; 0.0 = in group but not eligible; NaN = other groups
for task in ETM_TASKS:
    cohort[f'mdd_eligible_{task}'] = np.where(
        cohort['group'] == 'MDD',
        cohort['person_id'].isin(mdd_eligible_per_task[task]).astype(float),
        np.nan,
    )
    cohort[f'bd_eligible_{task}'] = np.where(
        cohort['group'] == 'BD',
        cohort['person_id'].isin(bd_eligible_per_task[task]).astype(float),
        np.nan,
    )

# -- 6h: Merge per-task washout + Fitbit flags (MDD and BD; HC gets NaN) ------
cohort = cohort.merge(flags_df, on='person_id', how='left')

# -- 6h-hc: fitbit_sufficient_{task} for HC (no washout flags for HC) ----------
for task in ETM_TASKS:
    hc_mask   = cohort['group'] == 'HC'
    task_date = cohort[f'etm_date_{task}']
    completed = hc_mask & task_date.notna()
    cohort.loc[completed, f'fitbit_sufficient_{task}'] = np.where(
        cohort.loc[completed, 'fitbit_start'].notna(),
        (cohort.loc[completed, 'fitbit_start']
         <= task_date[completed] - pd.Timedelta(days=N_DAYS_WEARABLE)).astype(float),
        0.0,  # completed task but no Fitbit data → not sufficient
    )

# -- 6i: Merge per-task episode history (MDD and BD; HC gets NaN) -------------
cohort = cohort.merge(all_episode_df, on='person_id', how='left')

# -- 6j: Enforce column order per schema -------------------------------------
id_cols           = ['person_id', 'group']
mdd_eligible_cols = [f'mdd_eligible_{t}' for t in ETM_TASKS]
bd_eligible_cols  = [f'bd_eligible_{t}'  for t in ETM_TASKS]
fitbit_cols       = ['fitbit_start', 'fitbit_end']
etm_cols          = ['etm_date_first'] + [f'etm_date_{t}' for t in ETM_TASKS]
score_cols        = [
    'gradcpt_dprime',
    'flanker_rcs_interference',
    'delaydiscounting_lnk',
    'emorecog_positive_bias',
    'emorecog_avg_accuracy',
    'emorecog_happy_accuracy', 'emorecog_sad_accuracy',
    'emorecog_fearful_accuracy', 'emorecog_angry_accuracy',
]
flag_cols_all = (
    [f'gradcpt_{f}'          for f in ETM_SCHEMA['gradcpt']['flags']] +
    [f'delaydiscounting_{f}' for f in ETM_SCHEMA['delaydiscounting']['flags']] +
    [f'flanker_{f}'          for f in ETM_SCHEMA['flanker']['flags']] +
    [f'emorecog_{f}'         for f in ETM_SCHEMA['emorecog']['flags']]
)
per_task_flag_cols = [
    f'{flag}_{task}'
    for task in ETM_TASKS
    for flag in ['etm_washout_pass', 'fitbit_sufficient', 'wearable_washout_pass']
]
episode_cols = (
    [f'n_prior_episodes_{t}'          for t in ETM_TASKS] +
    [f'n_prior_dep_psychosis_eps_{t}' for t in ETM_TASKS] +
    [f'n_prior_bp_psychosis_eps_{t}'  for t in ETM_TASKS]
)
touch_cols = [f'touch_{t}' for t in ETM_TASKS]

ordered_cols = (id_cols + mdd_eligible_cols + bd_eligible_cols + fitbit_cols + etm_cols +
                score_cols + flag_cols_all + per_task_flag_cols + episode_cols + touch_cols)
extra_cols   = [c for c in cohort.columns if c not in ordered_cols]
if extra_cols:
    print(f'Unexpected extra columns (appended): {extra_cols}')

cohort = cohort[ordered_cols + extra_cols].reset_index(drop=True)

# -- 6k: Summary --------------------------------------------------------------
print('=== cohort_master summary ===')
print(f'Total rows : {len(cohort):,}')
print(f'MDD        : {(cohort["group"] == "MDD").sum():,}')
print(f'BD         : {(cohort["group"] == "BD").sum():,}')
print(f'HC         : {(cohort["group"] == "HC").sum():,}')

print('\nPer-task MDD eligibility (subset of MDD group):')
for task in ETM_TASKS:
    n_elig = int(cohort[f'mdd_eligible_{task}'].fillna(0).sum())
    print(f'  mdd_eligible_{task:<20}: {n_elig:,}')

print('\nPer-task BD eligibility (subset of BD group):')
for task in ETM_TASKS:
    n_elig = int(cohort[f'bd_eligible_{task}'].fillna(0).sum())
    print(f'  bd_eligible_{task:<20}: {n_elig:,}')

print('\nPer-task HC Fitbit sufficiency:')
for task in ETM_TASKS:
    n_suf = int(cohort.loc[cohort['group'] == 'HC', f'fitbit_sufficient_{task}'].fillna(0).sum())
    n_hc  = int((cohort['group'] == 'HC').sum())
    print(f'  fitbit_sufficient_{task:<20}: {n_suf:,} / {n_hc:,} HC')

print('\nPer-task touch-device coverage (EtM session metadata, all groups):')
for task in ETM_TASKS:
    col       = f'touch_{task}'
    n_missing = int(cohort[col].isna().sum())
    n_total   = len(cohort)
    print(f'  {col:<20}: {n_missing:,} missing / {n_total:,} total')

print('\nMissing scores (NaN = task not completed by participant):')
for col in score_cols:
    n_nan = cohort[col].isna().sum()
    print(f'  {col:<40}: {n_nan:,} ({100*n_nan/len(cohort):.1f}%)')

print('\n=== Full attrition cascade ===')
attrition_df = pd.DataFrame(attrition_rows)
display(attrition_df[['step', 'task', 'group', 'n_before', 'n_dropped', 'n_after', 'reason']])

# -- 6l: Save -----------------------------------------------------------------
# Always saved; OVERWRITE only gates the BigQuery pulls.
save_parquet(cohort, 'cohort/cohort_master.parquet')

atr_df = pd.DataFrame(attrition_rows)
save_parquet(atr_df, 'cohort/attrition.parquet')
print(f'Attrition log updated ({len(atr_df)} total entries) → cohort/attrition.parquet')
print(f'\nSchema:\n{cohort.dtypes.to_string()}')

In [ ]:
from scipy import stats

# Primary outcomes for visualisation and exploratory t-tests.
# emorecog_avg_accuracy = mean(happy, sad, fearful, angry) accuracy — EmoRecog's model outcome.
score_meta = [
    ('gradcpt_dprime',           'GradCPT\nd-prime',          'higher = better attention'),
    ('flanker_rcs_interference', 'Flanker\nRCS interference',  'higher = worse control'),
    ('delaydiscounting_lnk',     'Delay Discounting\nln(k)',   'higher = more impulsive'),
    ('emorecog_avg_accuracy',    'EmoRecog\navg accuracy',     'higher = better recognition'),
]
palette = {'MDD': '#d62728', 'BD': '#ff7f0e', 'HC': '#1f77b4'}

fig, axes = plt.subplots(1, len(score_meta), figsize=(16, 4))
for ax, (col, label, direction) in zip(axes, score_meta):
    for grp, color in palette.items():
        vals = cohort.loc[cohort['group'] == grp, col].dropna()
        vals.plot.kde(ax=ax, color=color, label=f'{grp} (N={len(vals):,})', lw=2)
        ax.axvline(vals.median(), color=color, ls='--', lw=1.2, alpha=0.8)
    ax.set_title(f'{label}\n({direction})', fontsize=9)
    ax.set_xlabel('')
    ax.legend(fontsize=8)
    ax.set_ylabel('Density')

plt.suptitle('Cognitive score distributions: MDD / BD / HC  (dashed = median)', y=1.02)
save_res(RES_OUT, 'score_distributions_by_group.png', overwrite=True, res=fig)
plt.tight_layout()
plt.show()

# Per-emotion accuracy breakdown (EmoRecog only)
emo_cols = [
    ('emorecog_happy_accuracy',   'Happy'),
    ('emorecog_sad_accuracy',     'Sad'),
    ('emorecog_fearful_accuracy', 'Fearful'),
    ('emorecog_angry_accuracy',   'Angry'),
]
fig2, axes2 = plt.subplots(1, 4, figsize=(16, 4))
for ax, (col, label) in zip(axes2, emo_cols):
    for grp, color in palette.items():
        vals = cohort.loc[cohort['group'] == grp, col].dropna()
        vals.plot.kde(ax=ax, color=color, label=f'{grp} (N={len(vals):,})', lw=2)
        ax.axvline(vals.median(), color=color, ls='--', lw=1.2, alpha=0.8)
    ax.set_title(f'EmoRecog\n{label} accuracy', fontsize=9)
    ax.set_xlabel('Proportion correct')
    ax.legend(fontsize=8)
    ax.set_ylabel('Density')

plt.suptitle('Per-emotion accuracy: MDD / BD / HC  (dashed = median)', y=1.02)
save_res(RES_OUT, 'emorecog_per_emotion_by_group.png', overwrite=True, res=fig2)
plt.tight_layout()
plt.show()

# Numeric summary + Welch t-test vs HC (exploratory only — the formal tests are in 04a)
print('=== Score summary (mean ± SD, median) and Welch t-test vs HC ===\n')
hc_vals_cache = {
    col: cohort.loc[cohort['group'] == 'HC', col].dropna()
    for col, _, _ in score_meta
}
for col, label, direction in score_meta:
    hc_v = hc_vals_cache[col]
    tag  = label.replace('\n', ' ')
    print(f'{tag} ({direction}):')
    print(f'  HC  (N={len(hc_v):3,}): {hc_v.mean():.3f} ± {hc_v.std():.3f}  med={hc_v.median():.3f}')
    for grp in ('MDD', 'BD'):
        grp_v = cohort.loc[cohort['group'] == grp, col].dropna()
        t, p  = stats.ttest_ind(grp_v, hc_v, equal_var=False)
        print(f'  {grp:<3} (N={len(grp_v):3,}): {grp_v.mean():.3f} ± {grp_v.std():.3f}  med={grp_v.median():.3f} | '
              f'Welch t={t:.2f}, p={p:.4f}')
    print()

In [ ]:
# Descriptive: % of MDD / BD participants with ≥1 prior episode, per episode column
target_cols = [
    'n_prior_episodes_gradcpt',
    'n_prior_episodes_delaydiscounting',
    'n_prior_episodes_flanker',
    'n_prior_episodes_emorecog',
    'n_prior_dep_psychosis_eps_gradcpt',
    'n_prior_dep_psychosis_eps_delaydiscounting',
    'n_prior_dep_psychosis_eps_flanker',
    'n_prior_dep_psychosis_eps_emorecog',
    'n_prior_bp_psychosis_eps_gradcpt',
    'n_prior_bp_psychosis_eps_delaydiscounting',
    'n_prior_bp_psychosis_eps_flanker',
    'n_prior_bp_psychosis_eps_emorecog'
]

clinical_df = cohort[cohort['group'].isin(['MDD', 'BD'])].copy()

binary_df = (clinical_df[target_cols] > 0).astype(int)
binary_df['group'] = clinical_df['group']

percentage_summary = (
    binary_df
    .groupby('group')
    .mean() * 100
)

percentage_summary_formatted = percentage_summary.T.map(lambda x: f"{x:.1f}%")
percentage_summary_formatted